# 01 -- Data Collection

Downloads and saves raw data from all sources:
- FRED macro series
- Commodities (Yahoo Finance)
- S&P 500 Index (CRSP)
- S&P 500 Constituents (CRSP stock-level)
- Financial Ratios (WRDS)

**Output:** Raw CSVs in `data/raw/`

In [1]:
import pandas as pd
import numpy as np
import os
import zipfile

PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
RAW_DIR = os.path.join(PROJECT_DIR, 'data', 'raw')
os.makedirs(RAW_DIR, exist_ok=True)

import sys
sys.path.insert(0, os.path.join(PROJECT_DIR, 'shared'))

## 1. FRED Macro Data

Each series is a CSV with `date` and `value` columns.
Store in `data/raw/macros/`.

Publ. lags (approx days): GDP ~90, Unemployment ~5, CPI ~15, Retail ~15, M1 ~20, Sentiment ~1

In [2]:
# FRED series catalog (from Gathering_Data.docx)
FRED_SERIES = {
    'bonds': 'IRLTLT01USQ156N', 'interest_rates': 'REAINTRATREARAT10Y',
    'inflation': 'CORESTICKM159SFRBATL', 'cpi': 'CPALTT01USM657N',
    'cpi_median': 'MEDCPIM158SFRBCLE',
    'gdp': 'GDPC1', 'gdp_per_capita': 'A939RX0Q048SBEA', 'gdp_monthly': 'BBKMGDP',
    'retail_sales': 'RSXFS', 'unemployment': 'UNRATE',
    'money_supply_m1': 'M1REAL', 'money_velocity': 'M1V',
    'consumer_sentiment': 'UMCSENT',
    'gov_spending': 'W068RCQ027SBEA', 'gov_deficit': 'M318501Q027NBEA',
    'gov_debt': 'GFDEBTN', 'corporate_earnings': 'BOGZ1FA106110115Q',
    'oil': 'DCOILWTICO', 'natural_gas': 'DHHNGSP', 'ppi': 'PPIACO',
    'electricity': 'APU000072610', 'vix': 'VIXCLS',
    'eur_usd': 'DEXUSEU', 'cny_usd': 'DEXCHUS',
    'wealth_inequality': 'WFRBST01134',
    'gdp_euro': 'CLVMNACSCAB1GQEA19', 'gdp_china': 'CHNLORSGPNOSTSAM',
    'cpi_euro': 'CPHPTT01EZM659N',
}

PUBLICATION_LAGS = {
    'gdp': 90, 'gdp_per_capita': 90, 'gdp_monthly': 30,
    'gdp_euro': 90, 'gdp_china': 90,
    'corporate_earnings': 60, 'gov_spending': 60, 'gov_deficit': 60,
    'gov_debt': 45, 'wealth_inequality': 90,
    'unemployment': 5, 'retail_sales': 15,
    'inflation': 15, 'cpi': 15, 'cpi_median': 15, 'cpi_euro': 30,
    'consumer_sentiment': 1, 'money_supply_m1': 20, 'money_velocity': 60,
    'electricity': 15, 'ppi': 15,
    'oil': 0, 'natural_gas': 0, 'vix': 0, 'bonds': 0,
    'interest_rates': 0, 'eur_usd': 0, 'cny_usd': 0,
}

print(f"Catalog: {len(FRED_SERIES)} FRED series")

Catalog: 28 FRED series


In [3]:
# Download via fredapi (set your key)
FRED_API_KEY = 'YOUR_KEY_HERE'
MACROS_DIR = os.path.join(RAW_DIR, 'macros')
os.makedirs(MACROS_DIR, exist_ok=True)

if FRED_API_KEY != 'YOUR_KEY_HERE':
    from fredapi import Fred
    fred = Fred(api_key=FRED_API_KEY)
    for name, sid in FRED_SERIES.items():
        print(f"Downloading {name} ({sid})...")
        try:
            data = fred.get_series(sid).reset_index()
            data.columns = ['date', 'value']
            data.to_csv(os.path.join(MACROS_DIR, f'{name}.csv'), index=False)
        except Exception as e:
            print(f"  ERROR: {e}")
    print("Done.")
else:
    print("Set FRED_API_KEY to download, or place CSVs in data/raw/macros/")

Set FRED_API_KEY to download, or place CSVs in data/raw/macros/


## 2. Commodities (Yahoo Finance)

In [4]:
COMMODITY_TICKERS = {
    'gold': 'GC=F', 'silver': 'SI=F', 'copper': 'HG=F', 'platinum': 'PL=F',
    'corn': 'ZC=F', 'wheat': 'ZW=F', 'soybeans': 'ZS=F',
    'coffee': 'KC=F', 'sugar': 'SB=F', 'cotton': 'CT=F',
    'utilities_etf': 'XLU', 'bond_futures': 'ZB=F', 'rates_10y': '^TNX',
}
COMMODITIES_DIR = os.path.join(RAW_DIR, 'commodities')
os.makedirs(COMMODITIES_DIR, exist_ok=True)

try:
    import yfinance as yf
    for name, ticker in COMMODITY_TICKERS.items():
        print(f"Downloading {name} ({ticker})...")
        data = yf.download(ticker, start='1990-01-01', progress=False)
        if not data.empty:
            df = data[['Close']].reset_index()
            df.columns = ['date', 'value']
            df.to_csv(os.path.join(COMMODITIES_DIR, f'{name}.csv'), index=False)
    print("Done.")
except ImportError:
    print("pip install yfinance, or place CSVs in data/raw/commodities/")

Failed to get ticker 'GC=F' reason: Expecting value: line 1 column 1 (char 0)

1 Failed download:
['GC=F']: Exception('%ticker%: No timezone found, symbol may be delisted')


Failed to get ticker 'SI=F' reason: Expecting value: line 1 column 1 (char 0)

1 Failed download:
['SI=F']: Exception('%ticker%: No timezone found, symbol may be delisted')


Failed to get ticker 'HG=F' reason: Expecting value: line 1 column 1 (char 0)

1 Failed download:
['HG=F']: Exception('%ticker%: No timezone found, symbol may be delisted')


Failed to get ticker 'PL=F' reason: Expecting value: line 1 column 1 (char 0)

1 Failed download:
['PL=F']: Exception('%ticker%: No timezone found, symbol may be delisted')


Failed to get ticker 'ZC=F' reason: Expecting value: line 1 column 1 (char 0)


KeyboardInterrupt: 

## 3. S&P 500 Index (CRSP)

Update `CRSP_ZIP_PATH` to your local file.

In [5]:
SP500_DIR = os.path.join(RAW_DIR, 'sp500')
os.makedirs(SP500_DIR, exist_ok=True)

# UPDATE THIS PATH
CRSP_ZIP_PATH = r'C:/Users/danie/Desktop/AlgoTrading/WRDS_DATASETS/CRSP [done]/4.1 Index Portfolios (Others)/Index - S&P500 Indexes/IndexFile SP/ohylyyogziey2bxi_csv.zip'

if os.path.exists(CRSP_ZIP_PATH):
    with zipfile.ZipFile(CRSP_ZIP_PATH, 'r') as z:
        target = [f for f in z.namelist() if f.endswith('.csv') or f.endswith('_csv')]
        if target:
            with z.open(target[0]) as f:
                sp500 = pd.read_csv(f)
            sp500['caldt'] = pd.to_datetime(sp500['caldt'])
            sp500 = sp500[sp500['caldt'] >= '1990-01-01']
            sp500 = sp500[['caldt', 'spindx', 'sprtrn']]
            sp500.to_csv(os.path.join(SP500_DIR, 'sp500_index.csv'), index=False)
            print(f"S&P 500: {len(sp500)} rows saved.")
else:
    print(f"Not found: {CRSP_ZIP_PATH}")
    print("Update path or place sp500_index.csv in data/raw/sp500/")

S&P 500: 8817 rows saved.


## 4. S&P 500 Constituents (CRSP stock-level)

Large file (~150M rows), processed in chunks.

In [10]:
CONSTITUENTS_DIR = os.path.join(RAW_DIR, 'constituents')
os.makedirs(CONSTITUENTS_DIR, exist_ok=True)

# UPDATE THIS PATH
CRSP_STOCKS_ZIP = r'C:/Users/danie/Desktop/AlgoTrading/WRDS_DATASETS/CRSP [done]/3. Index (V2 CIZ)/7. SP500 Constituents/n1rzhu7y7nz0bs7n_csv.zip'

COLS_KEEP = [
    'DlyCalDt', 'PERMNO', 'PERMCO', 'Ticker', 'SICCD', 'PrimaryExch',
    'ConditionalType', 'TradingStatusFlg', 'SecurityType', 'SecuritySubType',
    'SecurityActiveFlg', 'USIncFlg', 'ShareClass', 'IssuerType', 'ShareType',
    'DlyClose', 'DlyVol', 'DlyRet', 'DlyRetx', 'DlyCap', 'DlyPrevCap',
    'ShrOut', 'DlyNumTrd', 'sprtrn'
]

def filter_chunk(chunk):
    chunk['DlyCalDt'] = pd.to_datetime(chunk['DlyCalDt'], errors='coerce')
    chunk = chunk[chunk['DlyCalDt'] >= '2000-01-01']
    
    # Filters (adjusted for V2 CIZ column set)
    chunk = chunk[chunk['PrimaryExch'].isin(['Q', 'N'])]
    if 'ConditionalType' in chunk.columns:
        chunk = chunk[chunk['ConditionalType'] == 'RW']
    chunk = chunk[chunk['SecurityType'] == 'EQTY']
    chunk = chunk[chunk['SecuritySubType'] == 'COM']
    chunk = chunk[chunk['TradingStatusFlg'].isin(['A', 'D'])]
    chunk = chunk[chunk['IssuerType'] == 'CORP']
    # ShareClass doesn't exist in this format — skip
    
    chunk['market_cap'] = chunk['DlyClose'] * chunk['ShrOut'] * 1000
    
    # Keep only useful columns
    keep = ['DlyCalDt', 'PERMNO', 'PERMCO', 'Ticker', 'SICCD', 'PrimaryExch',
            'TradingStatusFlg', 'USIncFlg', 'DlyClose', 'DlyVol', 'DlyRet',
            'DlyRetx', 'DlyCap', 'ShrOut', 'DlyNumTrd', 'market_cap']
    avail = [c for c in keep if c in chunk.columns]
    return chunk[avail]

if os.path.exists(CRSP_STOCKS_ZIP):
    import time
    datasets = []
    with zipfile.ZipFile(CRSP_STOCKS_ZIP) as z:
        fname = z.namelist()[0]
        with z.open(fname) as f:
            for i, chunk in enumerate(pd.read_csv(f, chunksize=200_000, low_memory=False), 1):
                t0 = time.time()
                proc = filter_chunk(chunk)
                if not proc.empty:
                    datasets.append(proc)
                print(f"Chunk {i}: {len(proc)} rows ({time.time()-t0:.1f}s)")
    result = pd.concat(datasets, ignore_index=True)
    result.to_parquet(os.path.join(CONSTITUENTS_DIR, 'sp500_constituents.parquet'))
    print(f"Total: {len(result)} rows saved.")
else:
    print(f"Not found: {CRSP_STOCKS_ZIP}")

Chunk 1: 0 rows (0.1s)
Chunk 2: 0 rows (0.0s)
Chunk 3: 0 rows (0.0s)
Chunk 4: 0 rows (0.0s)
Chunk 5: 0 rows (0.0s)
Chunk 6: 0 rows (0.0s)
Chunk 7: 0 rows (0.0s)
Chunk 8: 0 rows (0.0s)
Chunk 9: 0 rows (0.0s)
Chunk 10: 0 rows (0.0s)
Chunk 11: 0 rows (0.0s)
Chunk 12: 0 rows (0.0s)
Chunk 13: 0 rows (0.0s)
Chunk 14: 0 rows (0.0s)
Chunk 15: 0 rows (0.0s)
Chunk 16: 0 rows (0.0s)
Chunk 17: 0 rows (0.0s)
Chunk 18: 0 rows (0.0s)
Chunk 19: 0 rows (0.0s)
Chunk 20: 0 rows (0.0s)
Chunk 21: 0 rows (0.0s)
Chunk 22: 0 rows (0.0s)
Chunk 23: 0 rows (0.0s)
Chunk 24: 0 rows (0.0s)
Chunk 25: 0 rows (0.0s)
Chunk 26: 0 rows (0.0s)
Chunk 27: 0 rows (0.0s)
Chunk 28: 0 rows (0.0s)
Chunk 29: 0 rows (0.0s)
Chunk 30: 0 rows (0.0s)
Chunk 31: 76187 rows (0.2s)
Chunk 32: 198709 rows (0.5s)
Chunk 33: 197048 rows (0.5s)
Chunk 34: 196448 rows (0.5s)
Chunk 35: 195818 rows (0.5s)
Chunk 36: 194745 rows (0.5s)
Chunk 37: 194391 rows (0.5s)
Chunk 38: 193998 rows (0.5s)
Chunk 39: 193661 rows (0.5s)
Chunk 40: 192559 rows (0.5s)


In [9]:
with zipfile.ZipFile(CRSP_STOCKS_ZIP) as z:
    fname = z.namelist()[0]
    with z.open(fname) as f:
        sample = pd.read_csv(f, nrows=5)
print(sorted(sample.columns.tolist()))

['CUSIP', 'ConditionalType', 'DlyAsk', 'DlyBid', 'DlyCalDt', 'DlyCap', 'DlyCapFlg', 'DlyClose', 'DlyCumFacPr', 'DlyCumFacShr', 'DlyDistRetFlg', 'DlyFacPrc', 'DlyHigh', 'DlyLow', 'DlyMMCnt', 'DlyNonOrdDivAmt', 'DlyNumTrd', 'DlyOpen', 'DlyOrdDivAmt', 'DlyPrc', 'DlyPrcFlg', 'DlyRet', 'DlyRetDurFlg', 'DlyRetMissFlg', 'DlyRetx', 'DlyVol', 'HdrCUSIP', 'INDNO', 'IssuerType', 'MbrEndDt', 'MbrStartDt', 'PERMCO', 'PERMNO', 'PrimaryExch', 'SICCD', 'SecuritySubType', 'SecurityType', 'ShareType', 'ShrOut', 'Ticker', 'TradingStatusFlg', 'USIncFlg']


## 5. Financial Ratios (WRDS)

In [8]:
# UPDATE THIS PATH
RATIOS_ZIP = r'C:/Users/danie/Desktop/AlgoTrading/WRDS_DATASETS/WRDS [done]/4. Financial Ratios Suite [d]/Company/Ratios.zip'

if os.path.exists(RATIOS_ZIP):
    with zipfile.ZipFile(RATIOS_ZIP, 'r') as z:
        target = [f for f in z.namelist() if f.endswith('.csv') or f.endswith('_csv')]
        if target:
            with z.open(target[0]) as f:
                ratios = pd.read_csv(f)
            for col in ratios.select_dtypes(include='object').columns:
                ratios[col] = ratios[col].astype(str)
            ratios.to_parquet(os.path.join(RAW_DIR, 'financial_ratios.parquet'))
            print(f"Ratios: {len(ratios)} rows saved.")
else:
    print(f"Not found: {RATIOS_ZIP}")

print("\n=== Data collection complete ===")

C:\Users\danie\AppData\Local\Temp\ipykernel_5380\1350829172.py:9: DtypeWarning: Columns (75) have mixed types. Specify dtype option on import or set low_memory=False.
  ratios = pd.read_csv(f)


Ratios: 3014102 rows saved.

=== Data collection complete ===
